# IN4080: obligatory assignment 1 (Autumn 2026)
 
Mandatory assignment 1 consists of three parts. In Part 1 (6 points), you will test and improve on a BPE (Byte-Pair-Encoding) tokenizer. In Part 2 (7 points), you will estimate an N-gram language model, based on a training corpus and the tokenizer you worked on in Part 1. Finally, in Part 3 (7 points), you will develop a basic classification model to distinguish between Bokmål and Nynorsk sentences.

You should answer all three parts. You are required to get at least 12/20 points to pass. The most important is that you try to answer each question (possibly with some mistakes), to help you gain a better and more concrete understanding of the topics covered during the lectures. There are also bonus questions for those of you who would like to deepen their understanding of the topics covered by this assignment.

- We assume that you have read and are familiar with IFI’s requirements and guidelines for mandatory assignments, see [here](https://www.uio.no/english/studies/examinations/compulsory-activities/mn-ifi-mandatory.html) and [here](https://www.uio.no/english/studies/examinations/compulsory-activities/mn-ifi-guidelines.html).
- The assignment must be submitted to [Devilry](https://devilry.ifi.uio.no/) by __Friday, September 11 at 23:59__ at the latest. Assignments submitted through other means (e.g. email) will not be considered.
- This is an individual assignment. You should not deliver joint submissions. 
- You may redeliver in Devilry before the deadline, but include all files in the last delivery. - Only the last delivery will be read! If you deliver more than one file, put them into a zip-archive. You don't have to include in your delivery the files already provided for this assignment. 
- Name your submission _your\_username\_in4080\_mandatory1_
- You can work on this assignment either on the IFI machines or on your own computer. 

*The preferred format for the assignment is a completed version of this Jupyter notebook*, containing both your code and explanations about the steps you followed. We want to stress that simply submitting code is __not__ by itself sufficient to complete the assignment - we expect the notebook to also contain explanations of what you have implemented, along with motivations for the choices you made along the way. Preferably use whole sentences, and mathematical formulas if necessary. Explaining in your own words (using concepts we have covered through in the lectures) what you have done and reflecting on your solution is an important part of the learning process - take it seriously!

Regarding the use of LLMs (ChatGPT or similar): you are allowed to use them as 'sparring partner', for instance to clarify something you have not understood. However, you are __not__ allowed to use them to generate solutions (either in part or in full) to the assignment tasks. 

__Technical tip__: Some of the tasks in this assignment will require you to extend methods in classes that are already partly implemented. To implement those methods directly in a Jupyter notebook, you can use the function `setattr` to attach a method to a given class: 

```python
class A:
    pass
a = A()

def foo(self):
    print('hello world!')
    
setattr(A, 'foo', foo)
```

First, make sure that all required modules are installed:

In [ ]:
# %pip install tqdm numpy scikit_learn

Note: you may need to restart the kernel to use updated packages.


## Part 1 : Tokenisation

We will start by building a basic tokenizer relying on white space and punctuation. 

__Task 1.1__ (2 points): Implement the method `split` below such that it takes a text as input and outputs a list of tokens. The tokenisation should simply be done by splitting on white space, except for punctuation markers and other symbols (`.,:;!?-()"`), which should correspond to their own token. For instance, the sentence "Pierre, who works at NR, also teaches at UiO." should be split into 12 tokens.

In [1]:
from typing import List
import re, string

# First we should define our punctuation markers, 
# I am using re.escaep, to brackets and hihyphens as a normal characters when we are using the regex
punc = f"[{re.escape(string.punctuation)}]"

# Define finction to find any punctuation markers in our tokens.
def find_punc(token, punc = punc):
    matches = re.findall(punc, token)
    # remove the marker from the token
    refined = re.sub(punc, "", token)
    return [refined, ]+ re.findall(punc, token)

def basic_tokenize(text: str) -> List[str]:
    """The method should split the text on white space, except for punctuation
    markers that should be considered as tokens of their own (even in the 
    absence of white space before or after their occurrence)"""

    # Implement here your basic tokenisation
    
    # First we should split the string by space
    tokens = text.split(' ')
    # Then search for any punctuation markers in each token
    tokens_list = list(map(lambda x: find_punc(x), tokens))
    tokens = [token for tokens in tokens_list for token in tokens]

    return tokens



In [2]:
# Test the implemented basic tokenizer
sample  = "Pierre, who works at NR, also teaches at UiO."
tokens = basic_tokenize(sample)
print(tokens)
print(len(tokens))
assert len(tokens) == 12, "Wrong implementation, it must be 12 tokens"

['Pierre', ',', 'who', 'works', 'at', 'NR', ',', 'also', 'teaches', 'at', 'UiO', '.']
12


We will now run the tokeniser on a small corpus, the [Norwegian Dependency Treebank](https://www.nb.no/sprakbanken/en/resource-catalogue/oai-nb-no-sbr-10/) (the corpus has been annotated with morphological features, syntactic functions and hierarchical structures, but we'll simply use here the raw text and discard all the annotation layers). We provide you with the data in the files `ndt_train_lm.txt` and `ndt_test_lm.txt`. 

__Task 1.2__ (1 point): Run the tokenizer you have implemented on `ndt_test_lm.txt`. How many tokens were extracted? And how many types (distinct words) were there? 

In [3]:
# The first question, how many tokens exist in the corpus (ndt_test_lm.txt)?
with open('ndt_test_lm.txt', 'r') as file_:
    tokens = basic_tokenize(file_.readlines()[0])
    print(len(tokens))

# The second question, how many unique tokens do we have? we can use python set to find it out.
unique_tokens = set(tokens)
print(len(unique_tokens))


260658
30747


We shall now use Byte-Pair Encoding (BPE) to limit the vocabulary of the tokenizer to 5,000.  An initial implementation of the algorithm is provided below.

In [ ]:
from typing import Dict, List, Tuple, Iterator
import numpy as np
from tqdm.notebook import tqdm

class BPETokenizer:
    """Tokenizer based on the Byte-Pair Encoding algorithm. 
    Note: the current implementation is limited to Latin characters (ISO-8859-1)"""

    def __init__(self, train_corpus_file: str, vocab_size = 5000, progress_bar_bool=True):
        """Creates a new BPE tokenizer, with merge pairs found using the given
        corpus file. The extraction of merge pairs stops when a vocabulary of 
        size vocab_size is reached."""

        # List of string pairs that should be merged when tokenizing
        # Example: ('e', 't'), which means that 'et' is a possible subword
        # Each string pair is mapped to an unique index number
        # (corresponding to their position in the self.vocab list)
        self.merge_pairs = {}

        # We add as basic vocab all characters of the extended ASCII
        self.vocab = [chr(i) for i in range(256)]
        self.progress_bar_bool = progress_bar_bool
        with open(train_corpus_file) as fd:

            # We first read the corpus, split on white space, and counts the
            # occurrences of each distinct word
            print("Counting word occurrences in corpus %s"%train_corpus_file, end="...", flush=True)
            text = fd.read()
            vocabulary_counts = {}
            for token in text.split():
                vocabulary_counts[token] = vocabulary_counts.get(token, 0) + 1
            print("Done")

            # We then iteratively extend the list of merge pairs until we
            # reach the desired size. Note: to speed up the algorithm, we 
            # extract n merge pairs at each iteration

            if progress_bar_bool: progress_bar = tqdm(total=vocab_size)
            while len(self.vocab) < vocab_size:
                most_common_pairs = self.get_most_common_pairs(vocabulary_counts)
                for common_pair in most_common_pairs:
                    self.merge_pairs[common_pair] = len(self.vocab)
                    self.vocab.append("".join(common_pair))
                if progress_bar_bool: progress_bar.update(len(most_common_pairs))
         #       print("Examples of new subwords:", ["".join(pair) for pair in most_common_pairs][:10])
            
    def get_most_common_pairs(self, vocabulary_counts: Dict[str,int], 
                              n:int=200) -> List[Tuple[str,str]]:
        """Given a set of distinct words along with their corresponding number 
        of occurrences in the corpus, returns the n most frequent pairs of subwords.       
        """

        # We count the frequencies of consecutive subwords in the vocabulary list
        pair_freqs = {}
        for word, word_count in vocabulary_counts.items():
            subwords = self.tokenize_word(word)
            for i in range(len(subwords)-1):
                byte_pair = (subwords[i], subwords[i+1])
                pair_freqs[byte_pair] = pair_freqs.get(byte_pair, 0) + word_count

        # And return the most frequent ones
        most_freq_pairs = sorted(pair_freqs.keys(), key=lambda x: pair_freqs[x])[::-1][:n]
        return most_freq_pairs

    def __call__(self, input:str, show_progress_bar=True) -> Iterator[str]:
        """Tokenizes a full text"""

        # We first split into whitespace-separated tokens, and then in subwords
        words = input.split()
        for word in tqdm(words) if show_progress_bar else words:
            subwords = self.tokenize_word(word)
            for subword in subwords:
                yield subword
                

    def tokenize_word(self, word):
        """Splits the word into subwords, according to the merge pairs 
        currently stored in self.merge_pairs."""

        # We start with a list of characters
        # (+ a final character to denote the end of the word)    
        splits = list(word) + [" "]

        # We continue until there is nothing left to be merged
        while len(splits)>=2:

            # We extract consecutive subword pairs
            pairs = [(splits[i], splits[i+1]) for i in range(len(splits)-1)]

            # We find the "best" pair of subwords to merge -- that is, the one with the 
            # lowest position in the list of merge rules
            best_pair_to_merge = min(pairs, key=lambda x: self.merge_pairs.get(x, np.inf))
            if best_pair_to_merge in self.merge_pairs:

                # We then merge the two subwords
                for i in range(len(splits)-1):
                    if (splits[i], splits[i+1]) == best_pair_to_merge:
                        merged_subword = self.vocab[self.merge_pairs[best_pair_to_merge]]
                        splits = splits[:i] + [merged_subword] + splits[i+2:]
                        break
            else:
                break
        return splits

__Task 1.3__ (1 point): Learn the BPE tokenizer on the `ndt_train_lm.txt` corpus, and then apply this tokenizer on `ndt_test_lm.txt`. Print the number of tokens and types (distinct subwords) obtained by this tokenizer on the test data. How do those numbers compare to the ones obtained with the basic tokenizer you had implemented earlier ? 

In [22]:
BPE_ndt = BPETokenizer('ndt_train_lm.txt')

Counting word occurrences in corpus ndt_train_lm.txt...Done


  0%|          | 0/5000 [00:00<?, ?it/s]

In [23]:
with open("ndt_test_lm.txt") as fd:
    text = fd.read()
    tokens = list(BPE_ndt(text))
    print(len(tokens), len(set(tokens)))


# Answer the asked question:
# Our basic tokenizer identified 260,658 tokens and 30,747 unique tokens, respectively, whereas the BPE Tokenizer identified 386,099 tokens and 4,408 unique tokens.
# The reason for this difference is obvious: the basic tokenizer splits text just by whitespace, 
# whereas the BPE algorithm iteratively merges the most frequent adjacent character sequences in the corpus.

  0%|          | 0/226096 [00:00<?, ?it/s]

386099 4408


__Task 1.4__ (2 points): The current BPE implementation is that it treats all characters in the same manner. A rather inconvenient side effect is that letters may be merged together with punctuation markers (like 'ing', ',' --> 'ing,'), if they are not separated by white space. Modify the implementation of the BPE algorithm above to prevent punctuation markers to be merged with letters. 

In [33]:
# I just change the tokenize method in BPETokenizer.
# For example where it's merging new chars to a subword, it should be checked whether that chars is punctuation markers or not.

class ModifiedBPETokenizer(BPETokenizer):
    def __init__(self, train_corpus_file, vocab_size=5000, progress_bar_bool=True):
        super().__init__(train_corpus_file, vocab_size, progress_bar_bool=progress_bar_bool)

    def tokenize_word(self, word):
    
        splits = list(word) + [" "]

        while len(splits)>=2:


            #### WHERE MODIFICATION HAPPENS:
            # pairs = [(splits[i], splits[i+1]) for i in range(len(splits)-1) ]
            ####
            pairs = list()
            for i in range(len(splits)-1):
                if splits[i] in string.punctuation or splits[i+1] in string.punctuation:
                    continue
                pairs.append((splits[i], splits[i+1]))
           
            if not pairs:
                break

            ####

            best_pair_to_merge = min(pairs, key=lambda x: self.merge_pairs.get(x, np.inf))

                
            if best_pair_to_merge in self.merge_pairs:
                for i in range(len(splits)-1):
                    if (splits[i], splits[i+1]) == best_pair_to_merge:
                        merged_subword = self.vocab[self.merge_pairs[best_pair_to_merge]]
                        splits = splits[:i] + [merged_subword] + splits[i+2:]
                        break
            else:
                break
        return splits

In [34]:
BPE_ndt = ModifiedBPETokenizer('ndt_train_lm.txt', progress_bar_bool=False)


Counting word occurrences in corpus ndt_train_lm.txt...Done


In [36]:
with open("ndt_test_lm.txt") as fd:
    text = fd.read()
    tokens = list(BPE_ndt(text))
    print(len(tokens), len(set(tokens)))

  0%|          | 0/226096 [00:00<?, ?it/s]

476042 2392


__Task 1.5__ (_optional, 2 extra points_): In a [tweet](https://x.com/karpathy/status/1759996551378940395) published last year, the well-known AI researcher Andrej Karpathy stressed that many of the current limitations of Large Language Models are actually a product of the tokenisation step. Explain at least 4 of the problems he mentioned in his tweet (you can of course search online, or watch Karpathy's own video lecture on tokenization).

## Part 2: N-gram language models

We will now train simple N-gram language models on the NDT corpus, using the tokenizers we have developed in Part 1.

Here is the skeleton of the code:

In [18]:
import numpy as np
from abc import abstractmethod
from collections import Counter

class LanguageModel:
    """Generic class for running operations on language models, using a BPE tokenizer"""

    def __init__(self, tokenizer: BPETokenizer):
        """Build an abstract language model using the provided tokenizer"""
        self.tokenizer = tokenizer
 
    @abstractmethod
    def predict(self, context_tokens: List[str]):
        """Given a list of context tokens (=previous tokens), returns a dictionary
          mapping each possible token to its probability"""
        raise NotImplementedError()
    
    @abstractmethod
    def get_perplexity(self, text: str):
        """Computes the perplexity of the given text according to the LM"""

        print("Tokenising input text:")
        tokens = list(self.tokenizer(text))
        
        print("Computing perplexity:")
        log_probs = 0
        for i in tqdm(range(len(tokens))):
            context_tokens = ["<s>"] + tokens[:i]
            predict_distrib = self.predict(context_tokens)

            # We add the log-probabilities
            log_probs += np.log(predict_distrib[tokens[i]])
            
        perplexity = np.exp(-log_probs/len(tokens))
        return perplexity

class NGramLanguageModel(LanguageModel):
    """Representation of a N-gram-based language model"""

    def __init__(self, training_corpus_file: str, tokenizer:BPETokenizer, ngram_size:int=3,
                  alpha_smoothing:float=1):
        """Initialize the N-gram model with:
        - a file path to a training corpus to estimate the N-gram probabilities
        - an already learned BPE tokenizer
        - an N-gram size
        - a smoothing parameter (Laplace smoothing)"""
        
        LanguageModel.__init__(self, tokenizer)
        self.ngram_size = ngram_size
        
        # We define a simple backoff distribution (here just a uniform distribution)
        self.default_distrib = {token:1/len(tokenizer.vocab) for token in tokenizer.vocab}

        # Dictionary mapping a context (for instance the two preceding words if ngram_size=3)
        # to another dictionary specifying the probability of each possible word in the 
        # vocabulary. The context should be a tuple of tokens.
        self.ngram_probs = {}
        with open(training_corpus_file) as fd:   

            # based on the training corpus, tokenizer, ngram-size and smoothing parameter,
            # fill the self.ngram_probs with the correct N-gram probabilities  
            text = fd.read()
            tokens = list(tokenizer(text))
            length = len(tokens)

            # To solve this task, first we should separate different windows as size as defined ngram_size
            # Here I separated windows by keeping our target
            n_grams = [tuple(tokens[i:i+self.ngram_size]) for i in range(length - self.ngram_size + 1)]
            n_grams_dist = Counter(n_grams)
            # Here I separated windows by not considering the last token in each window
            context = [tuple(tokens[i:i+self.ngram_size - 1]) for i in range(length - self.ngram_size + 1)]
            context_dist = Counter(context)


            self.vocab_size = len(tokenizer.vocab)

            # Here we are truing to find given a context window, what is the probability of each token.
            for window in tqdm(context_dist.keys()):

                context_count = context_dist.get(window, 0)

                dist_for_this_window = dict()
                for token in tokenizer.vocab:
                    full_gram = window + (token,)

                    ngram_count = n_grams_dist.get(full_gram, 0)

                    # We should calculate: P(w_next | context) = (Count(w_next, context) + alpha_smoothing) / (Count(context) + (alpha_smoothing + vacab_size))
                    prob = (ngram_count + alpha_smoothing) / (context_count + (alpha_smoothing * self.vocab_size))
                    dist_for_this_window[token] = prob

                self.ngram_probs[window] = dist_for_this_window


 

    def predict(self, context_tokens: List[str]):
        """Given a list of preceding tokens, returns the probability distribution 
        over the next possible token."""

        # We restrict the contextual tokens to (N-1) tokens
        context_tokens = tuple(context_tokens[-self.ngram_size+1:])

        # If the contextual tokens were indeed observed in the corpus, simply
        # returns the precomputed probabilities
        if context_tokens in self.ngram_probs:
            return self.ngram_probs[context_tokens]
        
        # Otherwise, we return a uniform distribution over possible tokens
        else:
            return self.default_distrib

__Task 2.1__ (6 points): Complete the initialization method `__init__` to estimate the correct N-gram probabilities (with smoothing) based on the corpus. Don't worry about making your implementation super-efficient (although you can if you wish).

In [ ]:
from collections import defaultdict, Counter

def __init__(self, training_corpus_file: str, tokenizer:BPETokenizer, ngram_size:int=2, alpha_smoothing=0.1):
        """Initialize the N-gram model with:
        - a file path to a training corpus to estimate the N-gram probabilities
        - an already learned BPE tokenizer
        - an N-gram size
        - a smoothing parameter (Laplace smoothing)"""
        LanguageModel.__init__(self, tokenizer)
        self.ngram_size = ngram_size
        
        # We define a simple backoff distribution (here just a uniform distribution)
        self.default_distrib = {token:1/len(tokenizer.vocab) for token in tokenizer.vocab}

        # Dictionary mapping a context (for instance the two preceding words if ngram_size=3)
        # to another dictionary specifying the probability of each possible word in the 
        # vocabulary. The context should be a tuple of tokens.
        self.ngram_probs = {}
        with open(training_corpus_file) as fd:  

            # ADD HERE YOUR CODE TO FILL THE VALUES IN self.ngram_probs
            text = fd.read()
            tokens = list(tokenizer(text))
            length = len(tokens)

            # To solve this task, first we should separate different windows as size as defined ngram_size
            # Here I separated windows by keeping our target
            n_grams = [tuple(tokens[i:i+self.ngram_size]) for i in range(length - self.ngram_size + 1)]
            n_grams_dist = Counter(n_grams)
            # Here I separated windows by not considering the last token in each window
            context = [tuple(tokens[i:i+self.ngram_size - 1]) for i in range(length - self.ngram_size + 1)]
            context_dist = Counter(context)


            self.vocab_size = len(tokenizer.vocab)

            # Here we are truing to find given a context window, what is the probability of each token.
            for window in tqdm(context_dist.keys()):

                context_count = context_dist.get(window, 0)

                dist_for_this_window = dict()
                for token in tokenizer.vocab:
                    full_gram = window + (token,)

                    ngram_count = n_grams_dist.get(full_gram, 0)

                    # We should calculate: P(w_next | context) = (Count(w_next, context) + alpha_smoothing) / (Count(context) + (alpha_smoothing + vacab_size))
                    prob = (ngram_count + alpha_smoothing) / (context_count + (alpha_smoothing * self.vocab_size))
                    dist_for_this_window[token] = prob

                self.ngram_probs[window] = dist_for_this_window



setattr(NGramLanguageModel, '__init__', __init__)

__Task 2.2__ (1 point): Train your language model in `ndt_train_lm.txt`, and compute its perplexity on the test data in `ndt_test_lm.txt`. The perplexity can be computed by calling the method `get_perplexity`. <br>
(_Note_: if the training takes too much time, feel free to stop the process after looking at a fraction of the corpus, at least while you are testing/developing your training setup).

In [21]:
ngram_lm = NGramLanguageModel('ndt_train_lm.txt', BPE_ndt)

  0%|          | 0/872824 [00:00<?, ?it/s]

  0%|          | 0/2566 [00:00<?, ?it/s]

In [22]:
with open('ndt_test_lm.txt', 'r') as file_:
    text = file_.read()
    perplexity = ngram_lm.get_perplexity(text)

print(perplexity)

Tokenising input text:


  0%|          | 0/226096 [00:00<?, ?it/s]

Computing perplexity:


  0%|          | 0/476042 [00:00<?, ?it/s]

100.92788710962127


__Task 2.3__ (_optional_, 4 bonus points): Improve the language model you have just developed. You can choose to focus on improving your model through a backoff mechanism, interpolation, or both. Once you are done, compute the perplexity again on the test corpus to ensure the language model has indeed improved.

In [ ]:
from collections import defaultdict, Counter

# I wanna apply interpolation
# In interpolation we calculate ngram_probs for each 1 to n_gram size, then combine them with some predefined weights.
def __init__(self, training_corpus_file: str, tokenizer:BPETokenizer, ngram_size:int=2, alpha_smoothing=0.1):

        LanguageModel.__init__(self, tokenizer)
        self.ngram_size = ngram_size
        self.tokenizer = tokenizer
        self.alpha_smoothing = alpha_smoothing
        self.default_distrib = {token:1/len(tokenizer.vocab) for token in tokenizer.vocab}
        self.ngram_probs_dict = {}
        
        for i in range(1, self.ngram_size+1):
             self.ngram_probs_dict[i] = self.calculate_ngram_prob(training_corpus_file, i)

def calculate_ngram_prob(self, training_corpus_file, ngram_size):
    ngram_probs = {}
    with open(training_corpus_file) as fd:  

        text = fd.read()
        tokens = list(self.tokenizer(text))
        length = len(tokens)

        # To solve this task, first we should separate different windows as size as defined ngram_size
        # Here I separated windows by keeping our target
        n_grams = [tuple(tokens[i:i+ ngram_size]) for i in range(length -  ngram_size + 1)]
        n_grams_dist = Counter(n_grams)
        # Here I separated windows by not considering the last token in each window
        context = [tuple(tokens[i:i+ ngram_size - 1]) for i in range(length -  ngram_size + 1)]
        context_dist = Counter(context)


        self.vocab_size = len(self.tokenizer.vocab)

        # Here we are truing to find given a context window, what is the probability of each token.
        for window in tqdm(context_dist.keys()):

            context_count = context_dist.get(window, 0)

            dist_for_this_window = dict()
            for token in self.tokenizer.vocab:
                full_gram = window + (token,)

                ngram_count = n_grams_dist.get(full_gram, 0)

                # We should calculate: P(w_next | context) = (Count(w_next, context) + alpha_smoothing) / (Count(context) + (alpha_smoothing + vacab_size))
                prob = (ngram_count + self.alpha_smoothing) / (context_count + (self.alpha_smoothing * self.vocab_size))
                dist_for_this_window[token] = prob

            ngram_probs[window] = dist_for_this_window

        return ngram_probs


def predict(self, context_tokens: List[str]):
    """Given a list of preceding tokens, returns the interpolated probability distribution 
    over the next possible token."""
    
    # Initialize an empty distribution with 0.0 for every token in the vocabulary
    combined_distrib = {token: 0.0 for token in self.tokenizer.vocab}
    
    # Using uniform weights for interpolation 
    weight = 1.0 / self.ngram_size
    
    # Loop through each n-gram size we calculated (e.g., 1 and 2)
    for current_n in range(1, self.ngram_size + 1):
        
        # The context window size for an n-gram is n - 1
        context_length = current_n - 1
        
        # Get the context tuple 
        if context_length == 0:
            current_context = ()
        else:
            current_context = tuple(context_tokens[-context_length:])
            
        # Fetch the probability distribution for this specific n-gram level
        if current_context in self.ngram_probs_dict[current_n]:
            distrib = self.ngram_probs_dict[current_n][current_context]
        else:
            distrib = self.default_distrib
            
        # Add the weighted probabilities to the combined distribution
        for token in self.tokenizer.vocab:
            combined_distrib[token] += weight * distrib.get(token, self.default_distrib[token])
            
    return combined_distrib

setattr(NGramLanguageModel, '__init__', __init__)
setattr(NGramLanguageModel, 'calculate_ngram_prob', calculate_ngram_prob)
setattr(NGramLanguageModel, 'predict', predict)

In [30]:
ngram_lm = NGramLanguageModel('ndt_train_lm.txt', BPE_ndt, ngram_size=2)


  0%|          | 0/872824 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/872824 [00:00<?, ?it/s]

  0%|          | 0/2566 [00:00<?, ?it/s]

In [31]:
with open('ndt_test_lm.txt', 'r') as file_:
    text = file_.read()
    perplexity = ngram_lm.get_perplexity(text)

print(perplexity)

Tokenising input text:


  0%|          | 0/226096 [00:00<?, ?it/s]

Computing perplexity:


  0%|          | 0/476042 [00:00<?, ?it/s]

153.60844742198086


Although I used interpolation, the perplexity is higher. Which shows without that the situation was better (Compared to 100).
But the main reason for this is the weights that I used.
Here, weights are distributed uniformly; With tuning this parameters, we will be able to achieve better result. For example. considering higher weights for more ngram_size.

## Part 3: Text classification

We will finally use the texts from the Norwegian Dependency Treebank for a classification task -- more precisely to determine whether a sentence is likely to be written in Bokmål or Nynorsk. To this end, we will use a simple bag-of-word setup (or more precisely a bag-of-_subwords_, since we will rely on the subwords extracted using BPE) along with a logistic regression model. As there is only two, mutually exclusive classes, you can view the task as a binary classification problem. 

The training data is found in `ndt_train_class.txt` and simply consists of a list of sentences, each sentence being mapped to a language form (Bokmål: `nob` or Nynorsk: `nno`). The language form is written at the end of each line, separated by a `\t`. Note the training examples are currently _not_ shuffled.

To train and apply your classifier, the easiest is to use the [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) model from `scikit-learn`.

__Task 3.1__ (2 points): Create a `N x V` matrix in which each row corresponds to a training example (out of `N` training instances) and each column corresponds to an individual feature, in this case the presence/absence of a particular subword in the sentence. In other words, there should be a total of `V` features, where `V` is the total size of the vocabulary for our BPE tokenizer. Also create a vector of length `N` with a value of `1` if the sentence was marked as Nynorsk, and 0 if is was marked as Bokmål. 

In [ ]:
import numpy as np

# Here, first I splited the text and target as label.
# Then I applied tokenizer for each line
# I considered each line as a sample, and based on whether a token exists or not, put zero or one in the vector related to the sample.
def dataset_reader(path='ndt_train_class.txt', tokenizer=BPE_ndt):
    with open(path, 'r') as file_:
        lines = file_.readlines()
        sample_size = len(lines)
        vocab_size = len(tokenizer.vocab)
        targets = list()
        data_mat = np.zeros((sample_size, vocab_size))

        # I created a dictionary containing tokens as keys and their related index as values.
        vocab_dict = {token:idx for idx, token in enumerate(tokenizer.vocab)}

        for idx, line in enumerate(lines):
            text, target = line.split('\t')
            target = target.strip() 
            text = text.strip()
            
            # Map Nynorsk ('nno') to 1 and Bokmål ('nob') to 0
            if target == 'nno':
                targets.append(1)
            else:
                targets.append(0)
                
            tokens = tokenizer(text, show_progress_bar=False)
            for token in tokens:
                token_id = vocab_dict.get(token, None)
                if token_id:
                    data_mat[token_id, token_id] = 1



    targets = np.array(targets).reshape(-1,1)

    return data_mat, targets


In [38]:
seq, targets = dataset_reader()

__Task 3.2__ (2 points): Use the data matrix you have just filled to train a logistic regression model (see the documentation on [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) for more details). We recommend to use the `liblinear` solver. 

In [40]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(solver='liblinear')
clf.fit(seq, targets)


/fp/homes01/u01/ec-armans/.venvImg/lib64/python3.9/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LogisticRegression(solver='liblinear')

__Task 3.3__ (1 point): Now apply the learned logistic regression model to the test set in `ndt_test_class.txt`, and evaluate its performance in terms of accuracy, recall and precision (you can use the functionalities in `sklearn.metrics` to compute those easily).

In [46]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
test_seq, test_targets = dataset_reader('ndt_test_class.txt')

print('---------Train---------')
preds = clf.predict(seq)
print(f'Accuracy: {accuracy_score(targets, preds)}')
print(f'Recall: {recall_score(targets, preds)}')
print(f'Precision: {precision_score(targets, preds)}\n')
print("Confusion Matrix\n",confusion_matrix(targets, preds))

print('\n---------Test---------')
test_preds = clf.predict(test_seq)
print(f'Accuracy: {accuracy_score(test_targets, test_preds)}')
print(f'Recall: {recall_score(test_targets, test_preds)}')
print(f'Precision: {precision_score(test_targets, test_preds)}\n')
print("Confusion Matrix\n", confusion_matrix(test_targets, test_preds))

---------Train---------
Accuracy: 0.5736833305467313
Recall: 0.08583106267029973
Precision: 1.0

Confusion Matrix
 [[15959     0]
 [12749  1197]]

---------Test---------
Accuracy: 0.5193778353856124
Recall: 0.15210801873794433
Precision: 0.46661031276415893

Confusion Matrix
 [[3455  631]
 [3077  552]]


__Task 3.4__ (2 points): Inspect the weights learned by your logistic regression model (in `coef_`) and find the 5 subwords that contribute _the most_ to the classification of the sentence in Nynorsk. Also find the 5 subwords that contribute the most to the classification of the sentence in Bokmål. Do those weights make sense, according to what you know about Bokmål and Nynorsk ? 

In [58]:
vocab_coef = list(zip(BPE_ndt.vocab, clf.coef_[0]))
vocab_coef.sort(key=lambda x: x[1])

The **Most Positive Coefficients** push the model to predict label one representing Nynorsk.

The **Most Negative Coefficients** push the model to predict label zero representing Bokmål.


In [69]:
Nynorsk_coefs = vocab_coef[:-6:-1]
Nynorsk_coefs

[('Fle', np.float64(0.4275803907311293)),
 ('arane', np.float64(0.4275803907311293)),
 ('kommunane ', np.float64(0.4275803907311293)),
 ('jone', np.float64(0.4275803907311293)),
 ('lova ', np.float64(0.4275803907311293))]

In [70]:
Bokmål_coefs = vocab_coef[:6]
Bokmål_coefs

[(' ', np.float64(-0.3741871234103011)),
 ('!', np.float64(-0.3741871234103011)),
 ('&', np.float64(-0.3741871234103011)),
 ("'", np.float64(-0.3741871234103011)),
 ('(', np.float64(-0.3741871234103011)),
 (')', np.float64(-0.3741871234103011))]